# Fast Binoculars scoring on Colab A100
This notebook is tuned for an A100 runtime. It stages input parquet files from Drive to local SSD, initializes Binoculars once, uses adaptive batch sizing, and writes scored outputs and summaries back to Drive.

In [ ]:
# Install once per fresh runtime, then restart the runtime before running the rest.
!pip uninstall -y transformers tokenizers binoculars -q || true
!pip install -U pip setuptools wheel
!pip install tokenizers==0.13.3
!pip install transformers==4.41.2
!pip install "git+https://github.com/ahans30/Binoculars.git" --no-deps -q
print("Install complete. Restart the runtime, then continue from the next cell.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import gc
import json
import math
import os
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from binoculars import Binoculars

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv,noheader

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# change inp/outp dir depending on which data to score
# INPUT_DIR = Path("/content/drive/MyDrive/news_data/post_2021_processed")
# OUTPUT_DIR = Path("/content/drive/MyDrive/news_data/post_2021_binoculars_scored_a100")

INPUT_DIR = Path("/content/drive/MyDrive/news_data/pre_2022_gemma2_2b_paraphrased")
OUTPUT_DIR = Path("/content/drive/MyDrive/news_data/pre_2022_gemma2_2b_binoculars_scored_a100")

# local SSD staging/caching for faster IO
LOCAL_INPUT_DIR = Path("/content/binoculars_input_cache")
LOCAL_OUTPUT_DIR = Path("/content/binoculars_output_cache")

# Binoculars settings
MODE = "accuracy"   # "accuracy" or "low-fpr"
INITIAL_BATCH_SIZE = 8
MIN_BATCH_SIZE = 2
MIN_WORDS = 20

# thresholds from the project repo / Binoculars defaults
THRESHOLD_ACCURACY = 0.9015310749276843
THRESHOLD_LOW_FPR = 0.8536432310785527

# Auto-detect text column from these candidates, in order
# TEXT_COLUMN_CANDIDATES = ["chunk_text", "text", "maintext"]
TEXT_COLUMN_CANDIDATES = ["ai_generated_text"]

# Optional testing limits
MAX_FILES = None
MAX_ROWS_PER_FILE = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "summaries").mkdir(parents=True, exist_ok=True)
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = THRESHOLD_ACCURACY if MODE == "accuracy" else THRESHOLD_LOW_FPR

print("Input dir:", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Mode:", MODE)
print("Threshold:", THRESHOLD)
print("Initial batch size:", INITIAL_BATCH_SIZE)

In [ ]:
# Initialize Binoculars ONCE and reuse it for every file.
cleanup_memory()
bino = Binoculars(mode=MODE)
print("Binoculars initialized.")

In [ ]:
def detect_text_column_from_schema(parquet_path: Path, candidates=TEXT_COLUMN_CANDIDATES) -> str:
    schema_names = pq.ParquetFile(parquet_path).schema.names
    for col in candidates:
        if col in schema_names:
            return col
    raise ValueError(
        f"No usable text column found. Looked for: {candidates}. "
        f"Available columns: {schema_names}"
    )

def get_columns_to_read(parquet_path: Path, text_column: str):
    available = set(pq.ParquetFile(parquet_path).schema.names)
    desired = [text_column, "source_domain", "date_publish", "month_key", "time_bucket", "url", "original_text"]
    return [c for c in desired if c in available]

def preprocess_df(df: pd.DataFrame, text_column: str, min_words: int = MIN_WORDS) -> pd.DataFrame:
    df = df.dropna(subset=[text_column]).copy()
    df[text_column] = df[text_column].astype(str)
    df["_temp_word_count"] = df[text_column].str.split().str.len()
    df = df[df["_temp_word_count"] >= min_words].copy()
    df = df.drop(columns=["_temp_word_count"])
    if MAX_ROWS_PER_FILE is not None:
        df = df.iloc[:MAX_ROWS_PER_FILE].copy()
    return df.reset_index(drop=True)

def stage_input_file(input_path: Path) -> Path:
    local_path = LOCAL_INPUT_DIR / input_path.name
    if not local_path.exists() or local_path.stat().st_size != input_path.stat().st_size:
        shutil.copy2(input_path, local_path)
    return local_path

def save_scored_outputs(df_scored: pd.DataFrame, summary: dict, input_path: Path):
    local_scored = LOCAL_OUTPUT_DIR / f"{input_path.stem}_scored.parquet"
    local_summary = LOCAL_OUTPUT_DIR / f"{input_path.stem}_summary.json"

    df_scored.to_parquet(local_scored, index=False)
    with open(local_summary, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

    drive_scored = OUTPUT_DIR / local_scored.name
    drive_summary = OUTPUT_DIR / "summaries" / local_summary.name
    shutil.copy2(local_scored, drive_scored)
    shutil.copy2(local_summary, drive_summary)

    return drive_scored, drive_summary

def score_dataframe_adaptive(
    df: pd.DataFrame,
    text_column: str,
    initial_batch_size: int,
    threshold: float,
    bino: Binoculars,
    min_batch_size: int = 2,
) -> pd.DataFrame:
    scores = []
    predictions = []
    batch_sizes_used = []

    i = 0
    pbar = tqdm(total=len(df), desc="Scoring rows")
    while i < len(df):
        bs = min(initial_batch_size, len(df) - i)

        while True:
            batch_texts = df[text_column].iloc[i:i+bs].tolist()
            try:
                with torch.inference_mode():
                    batch_scores = bino.compute_score(batch_texts)

                if isinstance(batch_scores, (int, float, np.floating)):
                    batch_scores = [float(batch_scores)]

                batch_preds = [
                    "Most likely AI-generated" if s < threshold else "Most likely human-generated"
                    for s in batch_scores
                ]

                scores.extend(batch_scores)
                predictions.extend(batch_preds)
                batch_sizes_used.extend([bs] * len(batch_scores))
                i += len(batch_scores)
                pbar.update(len(batch_scores))
                break

            except RuntimeError as e:
                if "out of memory" in str(e).lower() and bs > min_batch_size:
                    cleanup_memory()
                    bs = max(min_batch_size, bs // 2)
                    print(f"OOM at row {i}; retrying with batch_size={bs}")
                    continue
                print(f"RuntimeError at row {i}: {e}")
                scores.extend([None] * len(batch_texts))
                predictions.extend(["Error"] * len(batch_texts))
                batch_sizes_used.extend([bs] * len(batch_texts))
                i += len(batch_texts)
                pbar.update(len(batch_texts))
                cleanup_memory()
                break
            except Exception as e:
                print(f"Error at row {i}: {e}")
                scores.extend([None] * len(batch_texts))
                predictions.extend(["Error"] * len(batch_texts))
                batch_sizes_used.extend([bs] * len(batch_texts))
                i += len(batch_texts)
                pbar.update(len(batch_texts))
                break

    pbar.close()

    result_df = df.copy()
    result_df["binoculars_score"] = scores
    result_df["binoculars_prediction"] = predictions
    result_df["batch_size_used"] = batch_sizes_used
    return result_df

def summarize_scored_df(df_scored: pd.DataFrame, source_file: str, text_column: str) -> dict:
    scored = pd.to_numeric(df_scored["binoculars_score"], errors="coerce")
    valid_scores = scored.dropna()

    ai_count = int((df_scored["binoculars_prediction"] == "Most likely AI-generated").sum())
    human_count = int((df_scored["binoculars_prediction"] == "Most likely human-generated").sum())
    error_count = int((df_scored["binoculars_prediction"] == "Error").sum())
    scored_count = int(scored.notna().sum())
    total_rows = int(len(df_scored))

    summary = {
        "source_file": source_file,
        "text_column": text_column,
        "mode": MODE,
        "threshold": THRESHOLD,
        "initial_batch_size": INITIAL_BATCH_SIZE,
        "min_words_filter": MIN_WORDS,
        "rows_scored_total": total_rows,
        "rows_scored_successfully": scored_count,
        "rows_with_errors": error_count,
        "ai_generated_count": ai_count,
        "human_generated_count": human_count,
        "ai_generated_pct": round(ai_count / scored_count * 100, 2) if scored_count else None,
        "score_mean": round(float(valid_scores.mean()), 6) if len(valid_scores) else None,
        "score_std": round(float(valid_scores.std()), 6) if len(valid_scores) > 1 else None,
        "score_min": round(float(valid_scores.min()), 6) if len(valid_scores) else None,
        "score_max": round(float(valid_scores.max()), 6) if len(valid_scores) else None,
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    }

    if "source_domain" in df_scored.columns:
        by_domain = []
        for name, grp in df_scored.groupby("source_domain", dropna=False):
            grp_scores = pd.to_numeric(grp["binoculars_score"], errors="coerce").dropna()
            grp_ai = int((grp["binoculars_prediction"] == "Most likely AI-generated").sum())
            grp_total = int(len(grp))
            by_domain.append({
                "source_domain": str(name),
                "total": grp_total,
                "ai_count": grp_ai,
                "human_count": int((grp["binoculars_prediction"] == "Most likely human-generated").sum()),
                "ai_pct": round(grp_ai / grp_total * 100, 2) if grp_total else None,
                "score_mean": round(float(grp_scores.mean()), 6) if len(grp_scores) else None,
            })
        summary["by_domain"] = sorted(by_domain, key=lambda x: x["total"], reverse=True)

    return summary

def process_one_file(input_path: Path, bino: Binoculars) -> dict:
    print(f"\n=== Processing {input_path.name} ===")
    t0 = time.time()

    local_input_path = stage_input_file(input_path)
    text_column = detect_text_column_from_schema(local_input_path)
    cols = get_columns_to_read(local_input_path, text_column)

    df = pd.read_parquet(local_input_path, columns=cols)
    rows_loaded = len(df)

    df_clean = preprocess_df(df, text_column=text_column, min_words=MIN_WORDS)
    rows_after_preprocess = len(df_clean)

    print(f"Loaded rows: {rows_loaded:,}")
    print(f"Rows after preprocessing: {rows_after_preprocess:,}")

    if rows_after_preprocess == 0:
        summary = {
            "source_file": input_path.name,
            "text_column": text_column,
            "mode": MODE,
            "threshold": THRESHOLD,
            "initial_batch_size": INITIAL_BATCH_SIZE,
            "min_words_filter": MIN_WORDS,
            "rows_loaded": rows_loaded,
            "rows_after_preprocessing": 0,
            "rows_scored_total": 0,
            "rows_scored_successfully": 0,
            "rows_with_errors": 0,
            "ai_generated_count": 0,
            "human_generated_count": 0,
            "ai_generated_pct": None,
            "score_mean": None,
            "score_std": None,
            "score_min": None,
            "score_max": None,
            "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        }
        return summary

    cleanup_memory()
    df_scored = score_dataframe_adaptive(
        df_clean,
        text_column=text_column,
        initial_batch_size=INITIAL_BATCH_SIZE,
        threshold=THRESHOLD,
        bino=bino,
        min_batch_size=MIN_BATCH_SIZE,
    )

    summary = summarize_scored_df(df_scored, source_file=input_path.name, text_column=text_column)
    summary["rows_loaded"] = rows_loaded
    summary["rows_after_preprocessing"] = rows_after_preprocess
    summary["runtime_seconds"] = round(time.time() - t0, 2)

    drive_scored, drive_summary = save_scored_outputs(df_scored, summary, input_path)
    summary["scored_output_path"] = str(drive_scored)
    summary["summary_output_path"] = str(drive_summary)

    print(f"Saved scored parquet to: {drive_scored}")
    print(f"Saved summary JSON to:  {drive_summary}")
    print(f"AI-generated: {summary['ai_generated_count']:,} / {summary['rows_scored_successfully']:,} ({summary['ai_generated_pct']}%)")
    print(f"Runtime: {summary['runtime_seconds']} sec")

    # free file-level CPU/GPU memory before next file
    del df, df_clean, df_scored
    cleanup_memory()

    return summary

In [ ]:
input_files = sorted(INPUT_DIR.glob("*.parquet"))
if MAX_FILES is not None:
    input_files = input_files[:MAX_FILES]

print(f"Found {len(input_files)} parquet files in {INPUT_DIR}")

summaries = []
for input_path in input_files:
    scored_output_path = OUTPUT_DIR / f"{input_path.stem}_scored.parquet"
    if scored_output_path.exists():
        print(f"Skipping {input_path.name} (already scored)")
        continue

    summary = process_one_file(input_path, bino)
    summaries.append(summary)

print("\nRun complete.")
print(f"Newly processed files this run: {len(summaries)}")

In [ ]:
summary_files = sorted((OUTPUT_DIR / "summaries").glob("*_summary.json"))
all_summaries = []

for path in summary_files:
    with open(path, "r", encoding="utf-8") as f:
        all_summaries.append(json.load(f))

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(OUTPUT_DIR / "all_file_summaries.csv", index=False)

display(summary_df.head())
print(f"Total summary files: {len(summary_df)}")

In [ ]:
if len(summary_df) == 0:
    print("No summary files found yet.")
else:
    total_loaded = int(summary_df["rows_loaded"].fillna(0).sum())
    total_after_pre = int(summary_df["rows_after_preprocessing"].fillna(0).sum())
    total_scored = int(summary_df["rows_scored_successfully"].fillna(0).sum())
    total_errors = int(summary_df["rows_with_errors"].fillna(0).sum())
    total_ai = int(summary_df["ai_generated_count"].fillna(0).sum())
    total_human = int(summary_df["human_generated_count"].fillna(0).sum())

    valid_file_means = pd.to_numeric(summary_df["score_mean"], errors="coerce").dropna()

    overall = {
        "input_dir": str(INPUT_DIR),
        "output_dir": str(OUTPUT_DIR),
        "mode": MODE,
        "threshold": THRESHOLD,
        "initial_batch_size": INITIAL_BATCH_SIZE,
        "min_words_filter": MIN_WORDS,
        "files_summarized": int(len(summary_df)),
        "rows_loaded_total": total_loaded,
        "rows_after_preprocessing_total": total_after_pre,
        "rows_scored_successfully_total": total_scored,
        "rows_with_errors_total": total_errors,
        "ai_generated_count_total": total_ai,
        "human_generated_count_total": total_human,
        "ai_generated_pct_total": round(total_ai / total_scored * 100, 2) if total_scored else None,
        "file_level_score_mean_mean": round(float(valid_file_means.mean()), 6) if len(valid_file_means) else None,
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    }

    overall_path = OUTPUT_DIR / "overall_summary.json"
    with open(overall_path, "w", encoding="utf-8") as f:
        json.dump(overall, f, indent=2, ensure_ascii=False, default=str)

    print(json.dumps(overall, indent=2))
    print(f"Saved overall summary to: {overall_path}")